In [0]:
import requests
import pandas as pd
from datetime import datetime, timedelta

In [0]:
# Job에서 전달된 watermark window
next_start_str = dbutils.jobs.taskValues.get(taskKey="check_lasttime", key="next_start")
next_end_str   = dbutils.jobs.taskValues.get(taskKey="check_lasttime", key="next_end")

next_start = datetime.fromisoformat(next_start_str)
next_end   = datetime.fromisoformat(next_end_str)

print(f"[INFO] BRZ processing window:")
print(f"       start = {next_start}")
print(f"       end   = {next_end}")

In [0]:
stations = (
    spark.table("hive_metastore.demo_airstatus_bronze.BRZ_seoul_stations")
    .select("stationName")
    .rdd
    .map(lambda r: r["stationName"])
    .collect()
)

print(f"[INFO] Number of stations: {len(stations)}")

In [0]:
SERVICE_KEY = "qz6MbARyTC9CL2GrNus/xLfgJolMh3LYaY+kT98k6wDcHDbTZ/vRAvw8JGmU9PnK25a7lM/ePpU1Dl7AsKAyXw=="
air_url = "http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty"


In [0]:
rows = []

for station in stations:
    params = {
        "serviceKey": SERVICE_KEY,
        "returnType": "json",
        "numOfRows": "100",
        "pageNo": "1",
        "stationName": station,
        "dataTerm": "DAILY",
        "ver": "1.0"
    }

    r = requests.get(air_url, params=params, timeout=10)
    if r.status_code != 200:
        print(f"❌ API failed ({station})")
        continue

    items = r.json()["response"]["body"]["items"]

    for i in items:
        raw_time = i.get("dataTime")
        if raw_time is None:
            continue

        if raw_time.endswith(" 24:00"):
            base_date = datetime.strptime(raw_time[:10], "%Y-%m-%d")
            dt = base_date + timedelta(days=1)
        else:
            dt = datetime.strptime(raw_time, "%Y-%m-%d %H:%M")

        if next_start <= dt <= next_end:
            rows.append({
                "stationName": station,
                "dataTime": raw_time,
                "khaiValue": i.get("khaiValue"),
                "khaiGrade": i.get("khaiGrade"),
                "pm10Value": i.get("pm10Value"),
                "pm25Value": i.get("pm25Value")
            })

print(f"[INFO] Collected rows in window: {len(rows)}")

if not rows:
    raise Exception("❌ No data collected for this window")

In [0]:
air_sdf = spark.createDataFrame(pd.DataFrame(rows))

air_sdf.write \
  .mode("overwrite") \
  .format("delta") \
  .saveAsTable("hive_metastore.demo_airstatus_bronze.BRZ_temp_seoul_air_quality_6h")

air_sdf.write \
  .mode("append") \
  .format("delta") \
  .saveAsTable("hive_metastore.demo_airstatus_bronze.BRZ_seoul_air_quality_hourly")